In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path('..').resolve()
FISHER_TAB = ROOT / 'results' / 'fisher_day0_day15' / 'tables'
POGO_TAB = ROOT / 'pogorelyy_repro' / 'results' / 'tables'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '09_sharing'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
MZ_PAIRS = [('P1', 'P2'), ('Q1', 'Q2'), ('S1', 'S2')]
MZ_SET = set(MZ_PAIRS) | {(b, a) for (a, b) in MZ_PAIRS}


In [ ]:
def jaccard(inter, a, b):
    u = a + b - inter
    return inter / u if u else np.nan

def load_fisher_aa():
    out = {}
    for d in DONORS:
        df = pd.read_csv(FISHER_TAB / f'fisher_positive_FC4_{d}.csv')
        out[d] = set(df['cdr3_aa'].astype(str))
    return out

def load_model_aa():
    out = {}
    for d in DONORS:
        df = pd.read_csv(POGO_TAB / f'{d}_15_top_expanded.tsv', sep='\t')
        out[d] = set(df['cdr3_aa'].astype(str))
    return out
fisher_aa = load_fisher_aa()
model_aa = load_model_aa()
{d: len(fisher_aa[d]) for d in DONORS}


In [ ]:
def pairwise_table(aa_by, method):
    rows = []
    for (a, b) in itertools.combinations(DONORS, 2):
        inter = len(aa_by[a] & aa_by[b])
        rows.append({'method': method, 'donor_a': a, 'donor_b': b, 'n_a': len(aa_by[a]), 'n_b': len(aa_by[b]), 'n_shared': inter, 'jaccard': jaccard(inter, len(aa_by[a]), len(aa_by[b])), 'is_mz_twin': (a, b) in MZ_SET})
    return pd.DataFrame(rows)
pair_f = pairwise_table(fisher_aa, 'Fisher')
pair_m = pairwise_table(model_aa, 'Pogorelyy')
pairs = pd.concat([pair_f, pair_m], ignore_index=True)
pairs.to_csv(TAB / 'pairwise_jaccard.csv', index=False)
pairs.head(12)


In [ ]:
def jaccard_matrix(aa_by):
    mat = pd.DataFrame(0.0, index=DONORS, columns=DONORS)
    for a in DONORS:
        for b in DONORS:
            if a == b:
                mat.loc[a, b] = 1.0
            else:
                inter = len(aa_by[a] & aa_by[b])
                mat.loc[a, b] = jaccard(inter, len(aa_by[a]), len(aa_by[b]))
    return mat
Mf = jaccard_matrix(fisher_aa)
Mm = jaccard_matrix(model_aa)
Mf.to_csv(TAB / 'jaccard_matrix_fisher.csv')
Mm.to_csv(TAB / 'jaccard_matrix_model.csv')
(fig, axes) = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(Mf, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0], vmin=0, vmax=0.02)

sns.heatmap(Mm, annot=True, fmt='.3f', cmap='YlGnBu', ax=axes[1], vmin=0, vmax=0.02)

plt.tight_layout()
plt.savefig(FIG / 'jaccard_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
breadth_rows = []
for (method, aa_by) in [('Fisher', fisher_aa), ('Pogorelyy', model_aa)]:
    counts = {}
    for (d, s) in aa_by.items():
        for aa in s:
            counts[aa] = counts.get(aa, 0) + 1
    for n_donors in range(1, 7):
        n_seq = sum((1 for v in counts.values() if v == n_donors))
        breadth_rows.append({'method': method, 'n_donors': n_donors, 'n_sequences': n_seq})
breadth = pd.DataFrame(breadth_rows)
breadth.to_csv(TAB / 'sharing_breadth.csv', index=False)
(fig, ax) = plt.subplots(figsize=(7, 4))
for (method, color) in [('Fisher', '#C0392B'), ('Pogorelyy', '#2980B9')]:
    sub = breadth[breadth['method'] == method]
    ax.plot(sub['n_donors'], sub['n_sequences'], marker='o', label=method, color=color)

ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'sharing_breadth.png', dpi=150, bbox_inches='tight')
plt.show()
print(pair_f[pair_f['is_mz_twin']][['donor_a', 'donor_b', 'jaccard', 'n_shared']].to_string(index=False))
